In [0]:
%python

# Databricks notebook source
# ==============================================================================
# PIPELINE BRONZE -> SILVER: LIMPEZA, PADRONIZAÇÃO E MERGE INCREMENTAL
# ==============================================================================

from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType
from delta.tables import DeltaTable

CATALOG = "credito_prd"
BRONZE_TABLE = f"{CATALOG}.bronze.give_me_some_credit_raw"
SILVER_TABLE = f"{CATALOG}.silver.give_me_some_credit"

# 1. Garante que o schema e o Volume de checkpoints existam no Unity Catalog
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.silver.checkpoints")

CHECKPOINT_SILVER = f"/Volumes/{CATALOG}/silver/checkpoints/give_me_some_credit/"

# 2. Leitura incremental da Bronze como Stream
df_bronze_stream = (
    spark.readStream
    .format("delta")
    .table(BRONZE_TABLE)
)

# 3. Transformações, Limpeza e Padronização de Colunas
df_silver_clean = (
    df_bronze_stream
    .withColumnRenamed("SeriousDlqin2yrs", "target_dlq_2yrs")
    .withColumnRenamed("RevolvingUtilizationOfUnsecuredLines", "revolving_utilization")
    .withColumnRenamed("age", "age")
    .withColumnRenamed("NumberOfTime30-59DaysPastDueNotWorse", "num_times_30_59_days_late")
    .withColumnRenamed("DebtRatio", "debt_ratio")
    .withColumnRenamed("MonthlyIncome", "monthly_income")
    .withColumnRenamed("NumberOfOpenCreditLinesAndLoans", "num_open_credit_lines")
    .withColumnRenamed("NumberOfTimes90DaysLate", "num_times_90_days_late")
    .withColumnRenamed("NumberRealEstateLoansOrLines", "num_real_estate_loans")
    .withColumnRenamed("NumberOfTime60-89DaysPastDueNotWorse", "num_times_60_89_days_late")
    .withColumnRenamed("NumberOfDependents", "num_dependents")
    .filter(F.col("customer_id").isNotNull())
    .filter((F.col("age") >= 18) & (F.col("age") <= 120))
    .withColumn("num_dependents", F.coalesce(F.col("num_dependents"), F.lit(0)).cast(IntegerType()))
    .withColumn("monthly_income", F.col("monthly_income").cast(DoubleType()))
    .withColumn("_silver_processed_at", F.current_timestamp())
)

# 4. Função de Micro-batch para aplicar Upsert (MERGE INTO) idempotente
def upsert_to_silver(batch_df, batch_id):
    batch_deduped = (
        batch_df
        .withColumn("_row_num", F.row_number().over(
            __import__("pyspark.sql.window").sql.Window.partitionBy("customer_id").orderBy(F.col("_ingestion_timestamp").desc())
        ))
        .filter(F.col("_row_num") == 1)
        .drop("_row_num")
    )

    if not spark.catalog.tableExists(SILVER_TABLE):
        batch_deduped.write.format("delta").mode("overwrite").saveAsTable(SILVER_TABLE)
    else:
        silver_delta = DeltaTable.forName(spark, SILVER_TABLE)
        (
            silver_delta.alias("tgt")
            .merge(
                batch_deduped.alias("src"),
                "tgt.customer_id = src.customer_id"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

# 5. Execução do Stream com trigger availableNow
query_silver = (
    df_silver_clean.writeStream
    .format("delta")
    .foreachBatch(upsert_to_silver)
    .option("checkpointLocation", CHECKPOINT_SILVER)
    .trigger(availableNow=True)
    .start()
)

query_silver.awaitTermination()

print(f"Carga Silver concluída com sucesso na tabela {SILVER_TABLE}!")

In [0]:
-- %python
-- # Databricks notebook source
-- # ==============================================================================
-- # PIPELINE BRONZE -> SILVER: LIMPEZA, PADRONIZAÇÃO E MERGE INCREMENTAL
-- # ==============================================================================

-- from pyspark.sql import functions as F
-- from pyspark.sql.types import IntegerType, DoubleType
-- from delta.tables import DeltaTable

-- CATALOG = "credito_prd"
-- BRONZE_TABLE = f"{CATALOG}.bronze.give_me_some_credit_raw"
-- SILVER_TABLE = f"{CATALOG}.silver.give_me_some_credit"
-- CHECKPOINT_SILVER = f"/Volumes/{CATALOG}/silver/checkpoints/give_me_some_credit/"

-- # 1. Garante a existência do schema silver
-- spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")

-- # 2. Leitura incremental da Bronze como Stream
-- df_bronze_stream = (
--     spark.readStream
--     .format("delta")
--     .table(BRONZE_TABLE)
-- )

-- # 3. Transformações, Limpeza e Padronização de Colunas
-- df_silver_clean = (
--     df_bronze_stream
--     # Padronização de nomes para snake_case
--     .withColumnRenamed("SeriousDlqin2yrs", "target_dlq_2yrs")
--     .withColumnRenamed("RevolvingUtilizationOfUnsecuredLines", "revolving_utilization")
--     .withColumnRenamed("age", "age")
--     .withColumnRenamed("NumberOfTime30-59DaysPastDueNotWorse", "num_times_30_59_days_late")
--     .withColumnRenamed("DebtRatio", "debt_ratio")
--     .withColumnRenamed("MonthlyIncome", "monthly_income")
--     .withColumnRenamed("NumberOfOpenCreditLinesAndLoans", "num_open_credit_lines")
--     .withColumnRenamed("NumberOfTimes90DaysLate", "num_times_90_days_late")
--     .withColumnRenamed("NumberRealEstateLoansOrLines", "num_real_estate_loans")
--     .withColumnRenamed("NumberOfTime60-89DaysPastDueNotWorse", "num_times_60_89_days_late")
--     .withColumnRenamed("NumberOfDependents", "num_dependents")
--     # Limpeza / Qualidade de Dados (Exemplo: flags e normalização)
--     .filter(F.col("customer_id").isNotNull())
--     .filter((F.col("age") >= 18) & (F.col("age") <= 120))
--     .withColumn("num_dependents", F.coalesce(F.col("num_dependents"), F.lit(0)).cast(IntegerType()))
--     .withColumn("monthly_income", F.col("monthly_income").cast(DoubleType()))
--     .withColumn("_silver_processed_at", F.current_timestamp())
-- )

-- # 4. Função de Micro-batch para aplicar Upsert (MERGE INTO) idempotente
-- def upsert_to_silver(batch_df, batch_id):
--     # Garante deduplicação dentro do próprio micro-batch caso existam IDs repetidos
--     batch_deduped = (
--         batch_df
--         .withColumn("_row_num", F.row_number().over(
--             __import__("pyspark.sql.window").sql.Window.partitionBy("customer_id").orderBy(F.col("_ingestion_timestamp").desc())
--         ))
--         .filter(F.col("_row_num") == 1)
--         .drop("_row_num")
--     )

--     # Cria a tabela caso não exista
--     if not spark.catalog.tableExists(SILVER_TABLE):
--         batch_deduped.write.format("delta").mode("overwrite").saveAsTable(SILVER_TABLE)
--     else:
--         silver_delta = DeltaTable.forName(spark, SILVER_TABLE)
--         (
--             silver_delta.alias("tgt")
--             .merge(
--                 batch_deduped.alias("src"),
--                 "tgt.customer_id = src.customer_id"
--             )
--             .whenMatchedUpdateAll()
--             .whenNotMatchedInsertAll()
--             .execute()
--         )

-- # 5. Execução do Stream com trigger availableNow (Micro-batch seguro e econômico)
-- query_silver = (
--     df_silver_clean.writeStream
--     .format("delta")
--     .foreachBatch(upsert_to_silver)
--     .option("checkpointLocation", CHECKPOINT_SILVER)
--     .trigger(availableNow=True)
--     .start()
-- )

-- query_silver.awaitTermination()

-- print(f"Carga Silver concluída com sucesso na tabela {SILVER_TABLE}!")